# Aargauer Bibliografie × swisscovery — Matching-Liste

Dieses Notebook erstellt eine Vorschlagsliste für neue deutschsprachige Wikipedia-Artikel:  
Personen/Musikgruppen aus dem [WikiProject Aargauer Bibliografie](https://www.wikidata.org/wiki/Wikidata:WikiProject_Aargauer_Bibliografie), die in swisscovery **mindestens zwei nicht-selbstverlegte Bücher** haben.

**Schritte:**
1. Setup & Imports
2. Wikidata-Fokusliste abrufen
3. swisscovery-Probe (3 Personen)
4. Vollabruf swisscovery (gecacht)
5. Matching anwenden
6. Export CSV + Markdown

## 1. Setup & Imports

In [ ]:
from __future__ import annotations
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in (_here, *_here.parents):
    if (_p / "src" / "aargau_match" / "__init__.py").exists():
        _s = str(_p / "src")
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from aargau_match.wikidata import fetch_focus_list
from aargau_match.swisscovery import search_by_gnd, search_by_name, BookRecord
from aargau_match.matching import build_person_result
from aargau_match.exporters import to_csv, to_markdown

DATA = Path('data')
DATA.mkdir(exist_ok=True)
(DATA / 'output').mkdir(exist_ok=True)

WIKIDATA_CSV = DATA / 'wikidata_persons.csv'
CACHE_FILE = DATA / 'swisscovery_hits.json'
OUT_CSV = DATA / 'output' / 'matching_liste.csv'
OUT_MD = DATA / 'output' / 'matching_liste.md'

## 2. Wikidata-Fokusliste abrufen

SPARQL gegen `query.wikidata.org`. ~2.000+ Personen (Stand 2026).

In [ ]:
if WIKIDATA_CSV.exists():
    persons = pd.read_csv(WIKIDATA_CSV, keep_default_na=False)
    print(f'Aus Cache geladen: {len(persons)} Personen')
else:
    persons = fetch_focus_list()
    persons.to_csv(WIKIDATA_CSV, index=False)
    print(f'Neu abgerufen: {len(persons)} Personen → {WIKIDATA_CSV}')

print(f'  davon mit GND-ID: {(persons.gnd != "").sum()}')
print(f'  davon mit dewiki-Artikel: {persons.has_dewiki.astype(bool).sum()}')
persons.head()

Aus Cache geladen: 2229 Personen
  davon mit GND-ID: 1359
  davon mit dewiki-Artikel: 434


,q_id,label,given,family,gnd,instance_of,birth,death,dewiki_url,has_dewiki,wikidata_url
0,Q100871456,Beat Gloor,Beat,Gloor,138582815,Mensch,1959-07-27,2020-02-06,https://de.wikipedia.org/wiki/Beat_Gloor,True,https://www.wikidata.org/wiki/Q100871456
1,Q100937370,Petra Young-zie Barthelmess Röthlisberger,Petra,Barthelmess,132119749,Mensch,1968-01-01,,,False,https://www.wikidata.org/wiki/Q100937370
2,Q102159116,Karl Matter,Karl,Matter,1060405423,Mensch,1874-01-01,1957-01-01,,False,https://www.wikidata.org/wiki/Q102159116
3,Q102225759,Thomas Fehlmann,Thomas,Fehlmann,130341061,Mensch,,,,False,https://www.wikidata.org/wiki/Q102225759
4,Q1022329,Christoph Baumann,Christoph,Baumann,135574773,Mensch,1954-06-26,,https://de.wikipedia.org/wiki/Christoph_Bauman...,True,https://www.wikidata.org/wiki/Q1022329


## 3. swisscovery-Probe (3 Personen)

Sanity-Check, dass GND- und Name-Suche funktionieren.

In [ ]:
# Stichproben (vor Vollabruf): Rollen-/Dedup-/Thesis-Logik prüfen.
SAMPLES = [
    {"q_id": "Q133448478", "label": "Maël Roumois", "gnd": "1249452813", "given": "Maël", "family": "Roumois"},
    {"q_id": "_blum", "label": "Robert Blum", "gnd": "", "given": "Robert", "family": "Blum"},
    {"q_id": "_kunz", "label": "Stephan Kunz", "gnd": "", "given": "Stephan", "family": "Kunz"},
    {"q_id": "_seiler", "label": "Hansjakob Seiler", "gnd": "", "given": "Hansjakob", "family": "Seiler"},
    {"q_id": "_bolliger", "label": "Max Bolliger", "gnd": "", "given": "Max", "family": "Bolliger"},
]
for s in SAMPLES:
    p = dict(s)
    row = persons[persons.label == p["label"]]
    if not row.empty:
        for k in ("q_id", "gnd", "given", "family"):
            v = str(row.iloc[0].get(k, "")).strip()
            if v:
                p[k] = v
    print(f"--- {p['label']} (GND={p['gnd'] or '–'}) ---")
    hits = search_by_gnd(str(p["gnd"])) if p["gnd"] else search_by_name(p["family"], p["given"])
    conf = "gnd" if p["gnd"] else "name-fuzzy"
    res = build_person_result(p, hits, confidence=conf)
    print(f"  raw: {len(hits)} | nach Rolle+Dedup: {res.books_total} | verified: {res.books_verified} | unverified: {res.books_unverified} | selfpub: {res.books_selfpub} | thesis: {res.books_thesis}")
    print(f"  qualifies={res.qualifies} review={res.requires_review} ({res.review_reason or '–'})")
    print(f"  Titel: {res.sample_titles[:300]}")


--- Klaus Merz (GND=118031198) ---
  raw hits: 200 | nach Creator-Filter+Dedup: 190
  non-selfpub: 190 | selfpub: 0 | qualifies: True
  Verlage (erste): Aargauer Kunsthaus; Aargauer Kuratorium; Aargauische Kantonalbank; Ammann; Artemis; Atlantis; Autoren Edition; Baumberger Print AG; Benteli; Büchergilde Gutenberg; Centre de traduction littéraire Univ


## 4. Vollabruf swisscovery

Für jede Person wird gesucht — primär per GND, sonst per Name. Treffer werden lokal in `data/swisscovery_hits.json` gecacht; ein Re-Run überspringt bereits abgerufene Personen.

Dauer: ca. 30–60 min beim ersten Lauf (abhängig von Rate-Limits & Anzahl Treffer).

> **Cache-Schema geändert** (Rollen je Creator): einmalig `data/swisscovery_hits.json` löschen und neu abrufen. Die Zelle unten erkennt und verwirft alten Cache automatisch.

In [ ]:
if CACHE_FILE.exists():
    cache = json.loads(CACHE_FILE.read_text())
else:
    cache = {}
# Schema-Guard: alter Cache (vor Rollen-Erfassung) ist inkompatibel.
_sample = next((h for v in cache.values() for h in v.get('hits', [])), None)
_sample_creator = (_sample or {}).get('creators', [{}])
_sample_creator = _sample_creator[0] if _sample_creator else {}
if _sample is not None and ('creators' not in _sample or 'dates' not in _sample_creator):
    print('Alter Cache erkannt (ohne Rollen/Lebensdaten) — wird verworfen und neu abgerufen.')
    cache = {}
print(f'Cache-Einträge: {len(cache)}')

todo = [r for _, r in persons.iterrows() if r.q_id not in cache]
print(f'Noch abzurufen: {len(todo)} / {len(persons)}')

FLUSH_EVERY = 50
for i, row in enumerate(tqdm(todo)):
    p = row.to_dict()
    qid = p['q_id']
    try:
        if p.get('gnd'):
            hits = search_by_gnd(str(p['gnd']))
            conf = 'gnd'
        elif p.get('family') or p.get('given'):
            hits = search_by_name(p.get('family',''), p.get('given',''))
            conf = 'name-fuzzy'
        else:
            hits, conf = [], 'none'
        cache[qid] = {
            'confidence': conf,
            'hits': [h.to_dict() for h in hits],
        }
    except Exception as exc:
        cache[qid] = {'confidence': 'error', 'hits': [], 'error': str(exc)}
    if (i + 1) % FLUSH_EVERY == 0:
        CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False))

CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False))
print(f'Fertig. Cache: {len(cache)} Einträge.')

Cache-Einträge: 0
Noch abzurufen: 2229 / 2229


  0%|          | 0/2229 [00:00<?, ?it/s]

## 5. Matching anwenden

Pro Person: Creator-vs-Subject-Filter, Selbstverlags-Heuristik, Qualifikation prüfen.

In [ ]:
results = []
for _, row in persons.iterrows():
    p = row.to_dict()
    entry = cache.get(p['q_id'], {'confidence': 'none', 'hits': []})
    hits = [BookRecord(**h) for h in entry['hits']]
    res = build_person_result(p, hits, confidence=entry.get('confidence', 'none'))
    results.append(res.as_row())
results_df = pd.DataFrame(results)
print(f'Gesamt: {len(results_df)}')
print(f'  qualifies=True: {results_df.qualifies.sum()}')
print(f'  davon ohne dewiki-Artikel: {((results_df.qualifies) & (~results_df.has_dewiki)).sum()}')
print(f'  davon requires_review: {((results_df.qualifies) & (results_df.requires_review)).sum()}')

fig, ax = plt.subplots(figsize=(8, 3))
results_df.books_non_selfpub.clip(upper=20).hist(bins=range(0, 22), ax=ax)
ax.set_xlabel('Anzahl nicht-selbstverlegte Bücher (≥20 = 20+)')
ax.set_ylabel('Personen')
ax.axvline(2, color='red', linestyle='--', label='Schwellwert (≥2)')
ax.legend()
plt.show()

## 6. Export CSV + Markdown

In [ ]:
csv_path = to_csv(results_df, OUT_CSV)
md_path = to_markdown(results_df, OUT_MD)
print(f'CSV: {csv_path}')
print(f'Markdown: {md_path}')

# Top-10-Kandidat/innen ohne dewiki-Artikel
top = results_df[(results_df.qualifies) & (~results_df.has_dewiki)].sort_values('books_non_selfpub', ascending=False)
top[['name','gnd','books_non_selfpub','confidence','requires_review','wikidata_url']].head(10)